# 03 - External Dataset Preparation

**Project:** DetectIt – Intelligent Web Threat Detection Using Machine Learning

**Goal:** A model can get a very high score on its own test set and still fail on new data. To check how our model works in the real world, we test it on a second dataset that was collected by different people at a different time: the [Web Page Phishing Detection Dataset](https://www.kaggle.com/datasets/shashwatwork/web-page-phishing-detection-dataset) (Hannousse & Yahiouche, 2021).

In this notebook, we clean this dataset, make sure it does not share any URLs with PhiUSIIL, and create the same URL features we made in `02_preprocessing.ipynb`. The result is used as the external test set in Experiment 2.

**Input:** `data/dataset_phishing.csv`, `data/PhiUSIIL_Phishing_URL_Dataset.csv`

**Output:** `data/processed/external_test.csv`, `data/processed/external_urls.csv`

## Setup

We load the libraries, our feature functions from `src/features.py`, and keep all settings in one place.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path("..").resolve()))
from src.features import build_url_features

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

EXTERNAL_PATH = Path("../data/dataset_phishing.csv")
PHIUSIIL_PATH = Path("../data/PhiUSIIL_Phishing_URL_Dataset.csv")
PROCESSED_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

TARGET = "label"
LABEL_NAMES = {0: "Legitimate", 1: "Phishing"}
CLASS_PALETTE = {"Legitimate": "#4C72B0", "Phishing": "#C44E52"}

In [2]:
ext = pd.read_csv(EXTERNAL_PATH)
initial_shape = ext.shape
print(f"Loaded external dataset: {initial_shape[0]:,} rows | {initial_shape[1]} columns")

Loaded external dataset: 11,430 rows | 89 columns


## 1. Dataset Overview

This dataset has a `url` column, 87 features and a `status` column (the label). Its 87 features are different from the PhiUSIIL features, so a model trained on PhiUSIIL cannot use them. Because of this, we only keep the `url` and `status` columns, and we create our own features from the URL.

In [3]:
ext[["url", "status"]].head()

,url,status
0,http://www.crestonwood.com/router.php,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,phishing
2,https://support-appleld.com.secureupdate.duila...,phishing
3,http://rgipt.ac.in,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,legitimate


In [4]:
print(f"Missing values: {ext.isnull().sum().sum()}")
print(f"Label values: {ext['status'].unique().tolist()}")

Missing values: 0
Label values: ['legitimate', 'phishing']
